# **CAUTION**
- This is an auxiliary step.
- The data is huge which is uploaded in teh AWS download the data entire data from AWS and put in the propser path
- Then reproduce
- Be care ful about paths (Rplace with your own path:  specifically this part: '/home/ayush/Desktop/GitHub-YAMA')


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

# **Source EEG based dataframe creation**
**Column names**

**BDSPPatientID, StudyID, Start Time Source File, SourceFilename, ErroneousBDSPInFile, ICANS, ICANSOnsetDate, NoOfDistinctSourceFiles**

In [ ]:
df_icans = pd.read_excel('/home/ayush/Desktop/GitHub-YAMA/ICANS/SLEEPHeadbasedSelectEEGSegments/Spreadsheets/DailyICANSData.xlsx')

df_icans_filenames = pd.read_excel('/home/ayush/Desktop/GitHub-YAMA/ICANS/SLEEPHeadbasedSelectEEGSegments/Spreadsheets/ICANSFiles.xlsx')
print('Total unique subjects by BDSPID =='+str(df_icans_filenames['BDSPPatientID'].nunique()))
print('Total unique subjects by StidyID =='+str(df_icans_filenames['StudyID'].nunique())+'\n')

# for i, col in enumerate(df_icans_filenames.columns):
#     print(i, col)
    
# ----------------------Select columns that start with 'Unnamed'--------------------------------
unnamed_cols = [col for col in df_icans_filenames.columns if col.startswith('Unnamed')]

# Check if all values in each column are NaN
# print('\n')
# for col in unnamed_cols:
#     all_nan = df_icans_filenames[col].isna().all()
#     print(f"{col}: All NaN? {all_nan}")

#------------------- Remove unnecessary columns from excel -------------------------
df_icans_filenames = df_icans_filenames.loc[:, ~df_icans_filenames.columns.str.startswith('Unnamed')]

#------------------- sort by Study IDs in assending order -------------------------
df_icans_filenames = df_icans_filenames.sort_values(by='StudyID', ascending=True).reset_index(drop=True)


#------------------------ Finding ICANS onset dates for each patient ----------------------------------
print('\n')
print('Finding ICANS onset dates for each patient')
df_icans['DateICANSOnset'] = pd.to_datetime(df_icans['DateICANSOnset'], errors='coerce')
# Group by StudyID, drop NaT, keep only date, join as comma-separated string
dates_per_study = (
    df_icans
    .groupby('StudyID')['DateICANSOnset']
    .apply(lambda x: ', '.join(x.dropna().dt.date.astype(str)))
    .reset_index()
)
dates_per_study.columns = ['StudyID', 'ICANS_Onset_Dates']

# Replace empty strings (no dates) with 'Unknown'
dates_per_study['ICANS_Onset_Dates'] = dates_per_study['ICANS_Onset_Dates'].replace('', 'Unknown')

# Count how many are unknown vs known
summary = dates_per_study['ICANS_Onset_Dates'].apply(lambda x: 'Unknown' if x == 'Unknown' else 'Known').value_counts()
total_patients = len(dates_per_study)
print(f"Total patients: {total_patients}\n")
print(summary)

# ----------------------------- Adding the ICANS Onset Date -----------------------------

# Create a dictionary mapping StudyID → ICANS_Onset_Dates
study_to_dates = dict(zip(dates_per_study['StudyID'], dates_per_study['ICANS_Onset_Dates']))

# Add new column to df_icans_filenames using the mapping
df_icans_filenames['ICANSOnsetDate'] = df_icans_filenames['StudyID'].map(study_to_dates)

def clean_icans_dates(dates_str):
    # If the value is NaN or Unknown, keep as is
    if pd.isna(dates_str) or dates_str == 'Unknown':
        return dates_str
    
    # Split by comma, remove extra spaces
    dates = [d.strip() for d in dates_str.split(',')]
    
    # Get unique dates
    unique_dates = sorted(set(dates))
    
    # Join back
    return ', '.join(unique_dates)

# Apply to the column
df_icans_filenames['ICANSOnsetDate'] = df_icans_filenames['ICANSOnsetDate'].apply(clean_icans_dates)

df_icans_filenames = df_icans_filenames.drop(
    columns=["Start Time Segments File", "Segments Filename"],
    errors="ignore"
)

# Finding unique source file per patient
df_icans_filenames["NoOfDistinctSourceFiles"] = (
    df_icans_filenames.groupby("StudyID")["SourceFilename"]
    .transform("nunique")
)

df_icans_filenames

# **Adding total No of vanilla valid 10 min segmensts from the SourceEEGFile for every patient**
**Added column names: 'VanillaValidSegmentsFilename', 'NoOfValidVanillaSegments'**

In [ ]:
base_path_to_vanilla_segments = Path(
    "/home/ayush/Desktop/GitHub-YAMA/ICANS/SLEEPHeadbasedSelectEEGSegments/AllContiniousEEGSegments/Morgoth_ready_vanillaEEGSegs/ICANS_group")

def get_segments_for_row(study_id, source_filename):
    # Extract timestamp from source filename
    match = re.search(r'(\d{4}-\d{2}-\d{2})_(\d{6})$', source_filename)
    if not match:
        return []
    
    date_part = match.group(1)
    time_part = match.group(2)
    time_formatted = f"{time_part[:2]}:{time_part[2:4]}:{time_part[4:]}"
    timestamp_for_seg = f"{date_part} {time_formatted}"
    
    # Glob all segments for this study in main folder
    all_seg_files = base_path_to_vanilla_segments.glob(f"{study_id}_seg*_SourceStart_*.mat")
    
    # Filter manually by timestamp and only keep the filename
    matched_files = []
    for f in all_seg_files:
        fname_clean = f.name.replace(":", "").replace(" ", "")
        timestamp_clean = timestamp_for_seg.replace(":", "").replace(" ", "")
        if timestamp_clean in fname_clean:
            matched_files.append(f.name)  # <--- only the filename, not full path
    
    return sorted(matched_files)

# Apply to the entire dataframe
df_icans_filenames['VanillaValidSegmentsFilename'] = df_icans_filenames.apply(
    lambda row: get_segments_for_row(row['StudyID'], row['SourceFilename']),
    axis=1
)

df_icans_filenames['NoOfValidVanillaSegments'] = df_icans_filenames['VanillaValidSegmentsFilename'].apply(len)

# Drop rows where NoOfValidVanillaSegments is 0
df_icans_filenames_filtered = df_icans_filenames[df_icans_filenames['NoOfValidVanillaSegments'] != 0].reset_index(drop=True)

df_icans_filenames_filtered

# **Selection criteria for the best 10 min segments that will be used to train the ICANS model**
- **Selection Criterian of Awake Snippets for ICANS = 0, 1, 2**
- **Selection Criterian of Awake Snippets for ICANS = 3, 4**

In [ ]:
csv_base_path = Path("/home/ayush/Desktop/GitHub-YAMA/ICANS/SLEEPHeadbasedSelectEEGSegments/AllContiniousEEGSegments/MORGOTH_SLEEP_OUTPUTS_to_select_top_snippets/ICANS_group")

#-----------------------------ICANS 0 1 2 ---------------------------------------
# The rows which has No of valid vanilla segmenst <=2; 
#   they have Best valid snippets filename same as vanilla_valid_segments_filename
def best_segments_icans012(row):
    segments = row['VanillaValidSegmentsFilename']
    segment_scores = []    
    for seg in segments:
        csv_name = seg.replace(".mat", ".csv")
        csv_path = csv_base_path / csv_name        
        if csv_path.exists():
            df_csv = pd.read_csv(csv_path)
            sleep_stage_pred = df_csv['pred_class']
            awake_score = (sleep_stage_pred == 0).sum() / len(sleep_stage_pred)
            
            segment_scores.append((seg, awake_score))  # store (segment, score)
        else:
            print("NOT FOUND:", csv_name)

    # Sort by awake_score descending
    segment_scores_sorted = sorted(segment_scores, key=lambda x: x[1], reverse=True)    
    # Take top 2 segment filenames
    top_2_segments = [seg for seg, score in segment_scores_sorted[:2]] 
    top_2_scores = [score for seg, score in segment_scores_sorted[:2]]
    return top_2_segments, top_2_scores

#-----------------------------ICANS 3 4 ---------------------------------------
def best_segments_icans34(row):
    segments = row['VanillaValidSegmentsFilename']
    segment_scores = []

    # Step 1: Calculate awake scores for all segments
    for seg in segments:
        csv_name = seg.replace(".mat", ".csv")
        csv_path = csv_base_path / csv_name

        if csv_path.exists():
            df_csv = pd.read_csv(csv_path)
            awake_score = (df_csv['pred_class'] == 0).sum() / len(df_csv)
            segment_scores.append((seg, awake_score, df_csv))
        else:
            print("NOT FOUND:", csv_name)
    
    # Step 2: Sort segments ascending by awake_score
    segment_scores_sorted = sorted(segment_scores, key=lambda x: x[1], reverse=True)

    # Take top 2 awake scores
    top_2_segments = segment_scores_sorted[:2]
    top_2_scores = [score for _, score, _ in top_2_segments]

    # Step 3: Decide which segments to keep
    if all(score >= 0.40 for score in top_2_scores):
        # Top 2 awake scores >= 40% → keep them
        selected_segments = [seg for seg, _, _ in top_2_segments]
    else:
        # Top 2 < 40% → use median of class_0_prob
        median_class0_list = []
        for seg, _, df_csv in segment_scores_sorted:
            median_class0 = df_csv['class_0_prob'].median()
            median_class0_list.append((seg, median_class0))

        # Sort by median_class0 descending
        median_class0_sorted = sorted(median_class0_list, key=lambda x: x[1], reverse=True)

        # Pick segment(s) with highest median
        max_median = median_class0_sorted[0][1]
        selected_segments = [seg for seg, med in median_class0_sorted if med == max_median]

    return selected_segments, top_2_scores

In [ ]:
from tqdm import tqdm
import pandas as pd
import numpy as np

# Step 1: Initialize BestValidSnippetsFilename for rows with <=2 segments
def select_best_segments(row):
    segs = row['VanillaValidSegmentsFilename']
    
    if len(segs) <= 2:
        return segs  # copy all segments
    else:
        return np.nan  # mark for processing later

df_icans_filenames_filtered['BestValidSnippetsFilename'] = df_icans_filenames_filtered.apply(
    select_best_segments,
    axis=1
)

# Step 2: Loop over all rows and process only empty/NaN rows
for i in tqdm(range(len(df_icans_filenames_filtered))):
    out_bestfilenames = df_icans_filenames_filtered.loc[i, 'BestValidSnippetsFilename']
    
    # Only process if NaN or empty list
    if (isinstance(out_bestfilenames, float) and pd.isna(out_bestfilenames)) or (isinstance(out_bestfilenames, list) and len(out_bestfilenames) == 0):
        row = df_icans_filenames_filtered.iloc[i]
        sid = row['StudyID']
        icans_val = row['ICANS']
        valid_segment_num = row['NoOfValidVanillaSegments']
        
        # For ICANS 0,1,2
        if (icans_val <= 2) and (valid_segment_num > 2):
            best_file_names, best_wake_scores = best_segments_icans012(row)
            
            print(f'Index {i}: Study ID {sid}: ICANS-{icans_val}: Top-2 wake scores == {best_wake_scores[0]*100:.2f}% and {best_wake_scores[1]*100:.2f}%')
            df_icans_filenames_filtered.at[i, 'BestValidSnippetsFilename'] = best_file_names
        
        # For ICANS 3,4
        elif (icans_val in [3, 4]) and (valid_segment_num > 2):
            best_file_names, best_wake_scores = best_segments_icans34(row)
            
            print(f'Index {i}: Study ID {sid}: ICANS-{icans_val}: Top-2 wake scores == {best_wake_scores[0]*100:.2f}% and {best_wake_scores[1]*100:.2f}%')
            # print(f'Selected segments for ICANS 3/4 == {best_file_names}')
            df_icans_filenames_filtered.at[i, 'BestValidSnippetsFilename'] = best_file_names

In [ ]:
# Count number of best segments per row
df_icans_filenames_filtered['NoOfBestSegments'] = df_icans_filenames_filtered['BestValidSnippetsFilename'].apply(
    lambda x: len(x) if isinstance(x, list) else 0
)

df_icans_filenames_filtered

# **Transfer the Best EEG snippets to another folder**

In [ ]:
# from tqdm import tqdm
# import shutil
# from pathlib import Path

# # Paths
# source_dir = Path("/home/ayush/Desktop/GitHub-YAMA/ICANS/SLEEPHeadbasedSelectEEGSegments/AllContiniousEEGSegments/Morgoth_ready_EEGSegs_ICANS/ICANS_group")
# dest_dir = Path("/home/ayush/Desktop/GitHub-YAMA/ICANS/ICANS10minEEGSegments")
# dest_dir.mkdir(parents=True, exist_ok=True)  # create destination folder if it doesn't exist

# # Loop over rows with progress bar
# for i, row in tqdm(df_icans_filenames_filtered.iterrows(), total=len(df_icans_filenames_filtered), desc="Copying best segments"):
#     best_segments = row['BestValidSnippetsFilename']
    
#     if isinstance(best_segments, list) and len(best_segments) > 0:
#         for seg_file in best_segments:
#             src_file = source_dir / seg_file
#             dst_file = dest_dir / seg_file
            
#             if src_file.exists():
#                 shutil.copy(src_file, dst_file)
#             else:
#                 print(f"Source file not found: {src_file}")